# Pre-Work: Customer Orders Data Model (Anh Huynh)

This notebook includes:

1. Data Model / ERD  
2. SQL Query: Most Sold Product Last Month  
3. Logical Architecture (Medallion / Databricks)

## 1. Data Model / ERD
### Table Definitions

#### customers
- customer_id (BIGINT, PK)
- customer_name (STRING)
- email (STRING)
- phone (STRING)
- created_at (TIMESTAMP)
- updated_at (TIMESTAMP)

#### products
- product_id (BIGINT, PK)
- product_name (STRING)
- sku (STRING)
- category (STRING)
- active_flag (BOOLEAN)
- created_at (TIMESTAMP)
- updated_at (TIMESTAMP)

#### orders
- order_id (BIGINT, PK)
- customer_id (BIGINT, FK → customers.customer_id)
- order_date (DATE)
- order_status (STRING)
- total_amount (DECIMAL)
- created_at (TIMESTAMP)
- updated_at (TIMESTAMP)

#### order_items
- order_item_id (BIGINT, PK)
- order_id (BIGINT, FK → orders.order_id)
- product_id (BIGINT, FK → products.product_id)
- line_number (INT)
- quantity (INT)
- unit_price (DECIMAL)
- line_status (STRING)
- status_updated_at (TIMESTAMP)
- created_at (TIMESTAMP)
- updated_at (TIMESTAMP)

### ERD Explanation (Short and Interview-Ready)

- **Customer → Orders**: One customer can have many orders (1:N).  
- **Orders → Order Items**: An order can have many line items (1:N).  
- **Products → Order Items**: A product can appear on many order items (1:N).  

This creates a clean star-like structure:

Customer  
   ↓  
Orders  
   ↓  
Order Items → Products  

**Item-level status** is tracked in `order_items.line_status`.  
**Order-level status** (`orders.order_status`) is typically **derived**:
- If all items are DELIVERED → the order is COMPLETED  
- If any item is still in transit → order remains OPEN / PROCESSING  

This dual-level status design allows:
- Monitoring partial shipments  
- Detecting bottlenecks  
- Accurate customer communication  
- Clean downstream analytics  


In [0]:
%sql
-- Create Schema
CREATE SCHEMA IF NOT EXISTS retail_demo;

USE retail_demo;

-- Customers Table
CREATE OR REPLACE TABLE customers (
    customer_id   BIGINT,
    customer_name STRING,
    email         STRING,
    phone         STRING,
    created_at    TIMESTAMP,
    updated_at    TIMESTAMP
) USING DELTA;

-- Products Table
CREATE OR REPLACE TABLE products (
    product_id    BIGINT,
    product_name  STRING,
    sku           STRING,
    category      STRING,
    active_flag   BOOLEAN,
    created_at    TIMESTAMP,
    updated_at    TIMESTAMP
) USING DELTA;

-- Orders Table
CREATE OR REPLACE TABLE orders (
    order_id      BIGINT,
    customer_id   BIGINT,
    order_date    DATE,
    order_status  STRING,
    total_amount  DECIMAL(18,2),
    created_at    TIMESTAMP,
    updated_at    TIMESTAMP
) USING DELTA;

-- Order Items Table
CREATE OR REPLACE TABLE order_items (
    order_item_id     BIGINT,
    order_id          BIGINT,
    product_id        BIGINT,
    line_number       INT,
    quantity          INT,
    unit_price        DECIMAL(18,2),
    line_status       STRING,
    status_updated_at TIMESTAMP,
    created_at        TIMESTAMP,
    updated_at        TIMESTAMP
) USING DELTA;


In [0]:
%sql
-- Insert sample customers
INSERT INTO customers VALUES
 (1, 'Alice', 'alice@example.com', '555-1111', current_timestamp(), current_timestamp()),
 (2, 'Bob',   'bob@example.com',   '555-2222', current_timestamp(), current_timestamp());

-- Insert sample products
INSERT INTO products VALUES
 (101, 'Widget A', 'WIDGET-A', 'Widgets', true, current_timestamp(), current_timestamp()),
 (102, 'Widget B', 'WIDGET-B', 'Widgets', true, current_timestamp(), current_timestamp()),
 (103, 'Gadget C', 'GADGET-C', 'Gadgets', true, current_timestamp(), current_timestamp());

-- Insert sample orders
INSERT INTO orders VALUES
 (1001, 1, DATE '2025-11-05', 'COMPLETED', 100.00, current_timestamp(), current_timestamp()),
 (1002, 1, DATE '2025-11-20', 'COMPLETED', 140.00, current_timestamp(), current_timestamp()),
 (1003, 2, DATE '2025-11-25', 'COMPLETED', 60.00,  current_timestamp(), current_timestamp()),
 (1004, 2, DATE '2025-10-15', 'COMPLETED', 80.00,  current_timestamp(), current_timestamp());  -- outside last month

-- Insert sample order_items
INSERT INTO order_items VALUES
 (1, 1001, 101, 1, 2, 10.00, 'DELIVERED', current_timestamp(), current_timestamp(), current_timestamp()),
 (2, 1001, 102, 2, 3, 20.00, 'DELIVERED', current_timestamp(), current_timestamp(), current_timestamp()),
 (3, 1002, 101, 1, 1, 10.00, 'DELIVERED', current_timestamp(), current_timestamp(), current_timestamp()),
 (4, 1002, 103, 2, 4, 15.00, 'DELIVERED', current_timestamp(), current_timestamp(), current_timestamp()),
 (5, 1003, 103, 1, 2, 15.00, 'DELIVERED', current_timestamp(), current_timestamp(), current_timestamp()),
 (6, 1004, 102, 1, 10, 20.00, 'DELIVERED', current_timestamp(), current_timestamp(), current_timestamp());  -- different month


## 2. SQL Query: Most Sold Product Last Month

In [0]:
%sql
WITH last_month_range AS (
  SELECT
    date_trunc('month', add_months(current_date(), -1)) AS month_start,
    date_trunc('month', current_date())                AS month_end
),

last_month_items AS (
  SELECT
      oi.order_id,
      oi.product_id,
      oi.quantity
  FROM order_items oi
  JOIN orders o
    ON o.order_id = oi.order_id
  CROSS JOIN last_month_range r
  WHERE
      o.order_date >= r.month_start
      AND o.order_date <  r.month_end
      AND oi.line_status = 'DELIVERED'
),

product_agg AS (
  SELECT
      p.product_id,
      p.product_name,
      SUM(lmi.quantity) AS total_quantity_sold
  FROM last_month_items lmi
  JOIN products p
    ON p.product_id = lmi.product_id
  GROUP BY
      p.product_id,
      p.product_name
),

ranked_products AS (
  SELECT
      product_id,
      product_name,
      total_quantity_sold,
      DENSE_RANK() OVER (ORDER BY total_quantity_sold DESC) AS rnk
  FROM product_agg
)

SELECT
    product_id,
    product_name,
    total_quantity_sold
FROM ranked_products
WHERE rnk = 1;


### Talking Points for Interview

- `last_month_range`: defines previous calendar month dynamically using `date_trunc` + `add_months`
- `last_month_items`: filters orders + order_items to that month and includes only `DELIVERED` items
- `product_agg`: groups by product and sums quantity sold
- `ranked_products`: uses `DENSE_RANK()` to handle ties and select the top-selling product(s)
- Query structure shows clear, step-wise logic similar to analytics engineering patterns (CTEs, layers, business rules)
- Easy to adapt if definition of "sold" changes or if top 3 products are needed

## 3. Logical Architecture (Medallion Approach)
### Medallion Architecture Overview

#### Bronze Layer (Raw Ingestion)
- Stores raw, unprocessed source data (orders, customers, products, order_items)
- Schema may not be enforced; includes semi-structured data (JSON/CSV)
- Acts as the immutable source of truth
- In production, ingestion could be batch or streaming (e.g., Kafka → Autoloader)

#### Silver Layer (Cleaned & Modeled)
- Where the four core tables in this project would live:
  - customers
  - products
  - orders
  - order_items
- Schema enforcement, deduplication, normalization
- Tracks referential integrity (order → order_items → products)
- Order-level and item-level statuses standardized and ready for analytics

#### Gold Layer (Analytics & Aggregates)
- Business-ready views and aggregates
- Examples:
  - product_sales_monthly
  - order_fulfillment_metrics
  - customer_lifetime_value
- Optimized for BI dashboards, KPIs, and stakeholder queries

#### Unity Catalog / Delta Lake
- All tables stored as Delta for ACID transactions & time travel
- UC controls permissions at catalog / schema / table level
- Supports lineage, quality checks, and governance


In [0]:
%sql
-- Preview tables (optional)

SELECT * FROM retail_demo.customers;
SELECT * FROM retail_demo.products;
SELECT * FROM retail_demo.orders;
SELECT * FROM retail_demo.order_items;


### End of Notebook

This notebook contains:
- Data model (ERD)
- Table definitions
- Delta table creation scripts
- Sample data
- CTE query for "most sold product last month"
- Medallion architecture explanation
- Interview talking points
